<a href="https://colab.research.google.com/github/ks148678-beep/Data-Warehousing-and-Data-Mining/blob/main/clean_retail_order_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [96]:
import pandas as pd
import numpy as np

DATA_PATH = "retail_orders_dirty (1).csv" # <-- only change this line

# Step 1: Inspect (correct way)
raw = pd.read_csv(DATA_PATH, dtype=str) # leave NA detection ON
df = raw.copy()

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing:\n", df.isna().sum())
print("\nFirst rows:\n", df.head().to_string())

# --- Cleaning starts ---

# 1. Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+","_", regex=True).str.strip("_")

Shape: (525, 11)

Columns: ['Order ID', ' Customer Name', 'Gender', 'AGE', 'City ', 'E-mail', 'Order_Date', 'Product Category', 'Qty', 'Unit Price (KES)', 'Total']

Missing:
 Order ID             0
 Customer Name       0
Gender              22
AGE                 30
City                23
E-mail              29
Order_Date          15
Product Category    12
Qty                  0
Unit Price (KES)    16
Total               22
dtype: int64

First rows:
     Order ID   Customer Name  Gender AGE    City                         E-mail  Order_Date Product Category Qty Unit Price (KES)  Total
0   ORD-1395  Faith Kiplagat  Female  18  Nairobi  faith.kiplagat84@example.com  2025-12-20        Groceries   1             1770   1770
1   ORD-1260     Ruth Otieno  Female  39  Mombasa     ruth.otieno40@example.com  29/06/2025        Groceries   3             1290   3870
2   ORD-1162    Brian Chebet       M  18   Nakuru    brian.chebet19@example.com  30/08/2025            books   3             2000   60

In [97]:
# 2. Remove duplicates on order_id
if "order_id" in df.columns:
    df["order_id"] = df["order_id"].str.upper().str.strip()
    df = df.drop_duplicates(subset=["order_id"], keep="first")

# 3. Clean text columns (product, customer, city, region)
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Replace n/a,?, - etc with NaN
df = df.replace(r'^\s*$', np.nan, regex=True)
df = df.replace(["n/a","NA","na","?","-","null","NULL","None"], np.nan)

In [98]:
# 4. Dates - handle mixed formats
for date_col in ["order_date","ship_date","delivery_date"]:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)

/tmp/ipykernel_2611/1717062900.py:4: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[date_col] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)


In [99]:
# 5. Numbers - remove $, commas, kg etc
for num_col in ["quantity","price","unit_price","sales","amount","discount"]:
    if num_col in df.columns:
        df[num_col] = df[num_col].astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
        df[num_col] = pd.to_numeric(df[num_col], errors="coerce")


In [100]:
# 6. Domain checks
if "quantity" in df.columns:
    df.loc[df["quantity"] <= 0, "quantity"] = np.nan
if "price" in df.columns:
    df.loc[df["price"] <= 0, "price"] = np.nan

In [101]:
# 7. Fix categories (city, region, category, payment_method)
if "city" in df.columns:
    df["city"] = df["city"].str.lower().str.title()
if "region" in df.columns:
    df["region"] = df["region"].str.lower().str.title()

In [102]:
# 8. Fill missing with median/mode
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include="object").columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "Unknown")

In [103]:
# 9. Save both CSV and Excel (Excel opens easier on phone)
df.to_csv("retail_orders_clean.csv", index=False)
df.to_excel("retail_orders_clean.xlsx", index=False)

print(f"\nCLEANED Shape: {df.shape} - Saved as retail_orders_clean.csv and.xlsx")
print("\nNow go to Files panel on left and Download the.xlsx file")


CLEANED Shape: (502, 11) - Saved as retail_orders_clean.csv and.xlsx

Now go to Files panel on left and Download the.xlsx file


In [104]:
from google.colab import files
df.to_excel("retail_orders_clean.xlsx", index=False)
files.download("retail_orders_clean.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [105]:
# Check how many emails ended up in order_date
print(df[df['order_date'].astype(str).str.contains('@', na=False)].head())

# Force order_date to be only dates - anything with @ becomes NaT (missing)
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', dayfirst=True)

# Now emails are removed from date column
print("Bad dates cleaned:", df['order_date'].isna().sum())

Empty DataFrame
Columns: [order_id, customer_name, gender, age, city, e_mail, order_date, product_category, qty, unit_price_kes, total]
Index: []
Bad dates cleaned: 183


In [106]:
# 1. Remove time 00:00:00 - keep only date
for col in ["order_date","ship_date","delivery_date","e_mail_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df[col] = df[col].dt.date # 2025-11-04 instead of 2025-11-04 00:00:00

# 2. Fill gaps - decide what to do with blank rows
# Option A: If city or e_mail is blank, put "Unknown"
df["city"] = df["city"].fillna("Unknown")
df["e_mail"] = df["e_mail"].fillna("unknown@example.com")
if "product_category" in df.columns:
    df["product_category"] = df["product_category"].fillna("Unknown")

# Option B: If you want NO gaps at all, delete rows where order_date is blank
df = df.dropna(subset=["order_date"]) # removes rows with empty dates

# 3. Remove completely empty rows
df = df.dropna(how="all")

print("Gaps left:", df.isna().sum().sum())
print("Final shape:", df.shape)

Gaps left: 0
Final shape: (319, 11)
